In [1]:
!nvidia-smi

Mon Aug 31 10:33:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:41:00.0 Off |                    0 |
| N/A   49C    P0            112W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install datasets transformers torch accelerate kernels tqdm


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

ds = load_dataset("EthanKim8683/reg_grpo", "128", split="train")
ds

Dataset({
    features: ['prompt', 'outputs', 'results'],
    num_rows: 128
})

In [4]:
import torch

def expand_groups(batch):
	new_batch = {
		"prompt": [],
		"output": [],
		"advantage": [],
	}
	for prompt, outputs, results in zip(
		batch["prompt"],
		batch["outputs"],
		batch["results"],
	):
		rewards = torch.tensor([1.0 if result else 0.0 for result in results])
		rewards_std = rewards.std(unbiased=False)
		if rewards_std < 1e-5:
			advantages = torch.zeros_like(rewards)
		else:
			advantages = (rewards - rewards.mean()) / rewards_std

		for output, advantage in zip(outputs, advantages):
			new_batch["prompt"].append(prompt)
			new_batch["output"].append(output)
			new_batch["advantage"].append(advantage)
	return new_batch

expanded_ds = ds.map(
	expand_groups,
	batched=True,
	remove_columns=ds.column_names
)
expanded_ds

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'output', 'advantage'],
    num_rows: 8192
})

In [5]:
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "openai/gpt-oss-20b"

model = AutoModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/410 [00:00<?, ?it/s]

[transformers] GptOssModel LOAD REPORT from: openai/gpt-oss-20b
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
class AdvHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.linear = torch.nn.Linear(hidden_size, 1, bias=False)

    def forward(self, x):
        x = self.linear(x)
        return torch.asinh(x)

adv_head = AdvHead(model.config.hidden_size)

In [ ]:
from tqdm import tqdm
from accelerate import Accelerator
import gc
from collections import deque

NUM_EPOCHS = 3

accelerator = Accelerator(mixed_precision="bf16")
device = accelerator.device

loss_fn = torch.nn.SmoothL1Loss(reduction="none")
optim = torch.optim.AdamW(adv_head.parameters(), lr=1e-4)

(
	model,
	adv_head,
	optim,
) = accelerator.prepare(
	model,
	adv_head,
	optim,
)

model.requires_grad_(False)
model.eval()

for epoch in range(NUM_EPOCHS):
	x_window = deque(maxlen=2**10)
	y_window = deque(maxlen=2**10)
	progress_bar = tqdm(
		expanded_ds.shuffle(),
		desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
		disable=not accelerator.is_main_process,
	)
	for example in progress_bar:
		optim.zero_grad()

		inputs = tokenizer.apply_chat_template(
			[
				{"role": "user", "content": example["prompt"]},
				{"role": "assistant", "content": example["output"]},
			],
			return_dict=True,
			return_tensors="pt",
			truncation=True,
			truncation_side="left",
			max_length=2**12,
		).to(device)

		with torch.no_grad():
			outputs = model(**inputs, use_cache=False)
			last_hidden_state = outputs.last_hidden_state
		
		preds = adv_head(last_hidden_state).squeeze(-1)
		targets = (
			torch.tensor(example["advantage"])
			.unsqueeze(-1)
			.expand_as(preds)
			.to(device)
		)
		loss = loss_fn(preds, targets).mean()
		
		accelerator.backward(loss)
		if accelerator.sync_gradients:
			accelerator.clip_grad_norm_(adv_head.parameters(), max_norm=1.0)
		optim.step()

		x_window.append(preds.flatten()[-1])
		y_window.append(targets.flatten()[-1])

		x = torch.tensor(x_window)
		y = torch.tensor(y_window)
		dx = x - x.mean()
		dy = y - y.mean()
		r = (dx * dy).sum() / (dx.norm() * dy.norm())

		progress_bar.set_postfix({
			"loss": f"{loss.item():.4f}",
			"r": f"{r.item():.4f}",
		})

		del (
			outputs,
			last_hidden_state,
			preds,
			targets,
			loss,
		)
		gc.collect()
		torch.cuda.empty_cache()

Epoch 1/3:   0%|          | 0/8192 [00:00<?, ?it/s]

Epoch 1/3:   0%|          | 9/8192 [00:07<1:42:14,  1.33it/s, loss=0.7240, r=0.2333] 